# Multi-Strategy Comparison

Compare RL training outcomes from different BC warm-start strategies.
Each run uses the same SAC hyperparameters but different initial policies:

- **Economic withholding** (`forecast_markup`): Cournot price markup at full volume
- **Physical withholding** (`cournot_physical_withholding`): MC price, capacity withheld

## Setup

Run the two scenarios with separate results directories:
```bash
python -m assume.testbed.learning_runner --scenario base_sac --results-dir testbed_results/rl_sac_economic
python -m assume.testbed.learning_runner --scenario base_sac_physical --results-dir testbed_results/rl_sac_physical
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ── Configuration ────────────────────────────────────────────────────────
STRATEGIES = {
    'Economic (forecast_markup)': '../testbed_results/rl_sac_economic',
    'Physical (cournot_physical)': '../testbed_results/rl_sac_physical',
}

MC = 35.0       # gen_0 marginal cost
GEN_ID = 'gen_0'

# Verify directories exist
for label, path in STRATEGIES.items():
    exists = os.path.isdir(path)
    print(f'{"OK" if exists else "MISSING":>7}  {label}: {path}')

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────
data = {}
for label, path in STRATEGIES.items():
    d = {}
    d['bids'] = pd.read_csv(f'{path}/bid_history.csv', parse_dates=['timestamp'])
    d['results'] = pd.read_csv(f'{path}/results_history.csv', parse_dates=['timestamp'])
    d['bids']['hour'] = d['bids']['timestamp'].dt.hour
    d['results']['hour'] = d['results']['timestamp'].dt.hour

    fp_path = f'{path}/final_policy_bids.csv'
    if os.path.exists(fp_path):
        d['final_bids'] = pd.read_csv(fp_path, parse_dates=['timestamp'])
        d['final_bids']['hour'] = d['final_bids']['timestamp'].dt.hour
    else:
        d['final_bids'] = None

    fr_path = f'{path}/final_policy_results.csv'
    if os.path.exists(fr_path):
        d['final_results'] = pd.read_csv(fr_path, parse_dates=['timestamp'])
        d['final_results']['hour'] = d['final_results']['timestamp'].dt.hour
    else:
        d['final_results'] = None

    data[label] = d
    n_episodes = d['bids']['episode'].nunique()
    print(f'{label}: {n_episodes} episodes, {len(d["bids"])} bid rows')

## 1. Training Convergence — Bid Price Evolution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, (label, d) in zip(axes, data.items()):
    bids = d['bids'][d['bids']['unit_id'] == GEN_ID]
    ep_stats = bids.groupby('episode')['bid_price'].agg(['mean', 'std', 'min', 'max'])

    ax.fill_between(ep_stats.index, ep_stats['min'], ep_stats['max'],
                    alpha=0.15, color='C0', label='min-max range')
    ax.fill_between(ep_stats.index,
                    ep_stats['mean'] - ep_stats['std'],
                    ep_stats['mean'] + ep_stats['std'],
                    alpha=0.3, color='C0', label='±1 std')
    ax.plot(ep_stats.index, ep_stats['mean'], 'o-', ms=2, color='C0', label='mean bid')
    ax.axhline(MC, color='C0', ls='--', alpha=0.5, label=f'MC = ${MC:.0f}')
    ax.set_xlabel('Episode')
    ax.set_title(label)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Bid Price ($/MWh)')
fig.suptitle(f'{GEN_ID} — Bid Price Convergence by Strategy', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Training Convergence — Bid Volume Evolution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, (label, d) in zip(axes, data.items()):
    bids = d['bids'][d['bids']['unit_id'] == GEN_ID]
    ep_stats = bids.groupby('episode')['bid_volume'].agg(['mean', 'std', 'min', 'max'])

    ax.fill_between(ep_stats.index, ep_stats['min'], ep_stats['max'],
                    alpha=0.15, color='C1', label='min-max range')
    ax.fill_between(ep_stats.index,
                    ep_stats['mean'] - ep_stats['std'],
                    ep_stats['mean'] + ep_stats['std'],
                    alpha=0.3, color='C1', label='±1 std')
    ax.plot(ep_stats.index, ep_stats['mean'], 'o-', ms=2, color='C1', label='mean volume')
    ax.axhline(300, color='gray', ls=':', alpha=0.5, label='max capacity')
    ax.set_xlabel('Episode')
    ax.set_title(label)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Bid Volume (MW)')
fig.suptitle(f'{GEN_ID} — Bid Volume Convergence by Strategy', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Final Policy Comparison — Hourly Bids

In [ ]:
fig, (ax_price, ax_vol) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

colors = ['C0', 'C1']
for i, (label, d) in enumerate(data.items()):
    fp = d['final_bids']
    if fp is None:
        print(f'{label}: no final policy bids')
        continue
    fp_gen = fp[fp['unit_id'] == GEN_ID].sort_values('hour')

    ax_price.plot(fp_gen['hour'], fp_gen['bid_price'], 'o-',
                  color=colors[i], label=label, ms=5)
    ax_vol.plot(fp_gen['hour'], fp_gen['bid_volume'], 's-',
               color=colors[i], label=label, ms=5)

ax_price.axhline(MC, color='gray', ls='--', alpha=0.5, label=f'MC = ${MC:.0f}')
ax_price.axhline(50, color='red', ls=':', alpha=0.4, label='gen_1 MC = $50')
ax_price.set_ylabel('Bid Price ($/MWh)')
ax_price.legend(fontsize=9)
ax_price.set_title(f'{GEN_ID} — Final Policy (noise-free)')

ax_vol.axhline(300, color='gray', ls=':', alpha=0.5, label='max capacity')
ax_vol.set_ylabel('Bid Volume (MW)')
ax_vol.set_xlabel('Hour')
ax_vol.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 4. Profit Comparison — Which Policy Earned More?

Compare actual profit (income - cost) from the final noise-free policy evaluation.

In [ ]:
fig, (ax_hour, ax_total) = plt.subplots(1, 2, figsize=(16, 5),
                                         gridspec_kw={'width_ratios': [3, 1]})

profit_totals = {}
for i, (label, d) in enumerate(data.items()):
    fr = d['final_results']
    if fr is None:
        print(f'{label}: no final results')
        continue
    gen = fr[fr['unit_id'] == GEN_ID].sort_values('hour')

    # Profit = (accepted_price - marginal_cost) * dispatch
    gen = gen.copy()
    gen['profit'] = (gen['accepted_price'] - gen['marginal_cost']) * gen['dispatch']

    hourly_profit = gen.groupby('hour')['profit'].sum()
    ax_hour.bar(hourly_profit.index + i*0.35 - 0.175, hourly_profit.values,
               width=0.35, color=colors[i], alpha=0.8, label=label)

    profit_totals[label] = hourly_profit.sum()

ax_hour.set_xlabel('Hour')
ax_hour.set_ylabel('Profit ($/h)')
ax_hour.set_title('Hourly Profit — Final Policy')
ax_hour.legend(fontsize=9)
ax_hour.set_xticks(range(24))

# Total profit bar chart
labels = list(profit_totals.keys())
values = list(profit_totals.values())
bars = ax_total.bar(range(len(labels)), values, color=colors[:len(labels)], alpha=0.8)
ax_total.set_xticks(range(len(labels)))
ax_total.set_xticklabels([l.split('(')[0].strip() for l in labels], fontsize=9)
ax_total.set_ylabel('Total Profit ($)')
ax_total.set_title('Total Daily Profit')
for bar, val in zip(bars, values):
    ax_total.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Print winner
if profit_totals:
    winner = max(profit_totals, key=profit_totals.get)
    print(f'\nWinner by total profit: {winner} (${profit_totals[winner]:,.0f})')

## 5. Dispatch Comparison — Who Gets Cleared?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for i, (label, d) in enumerate(data.items()):
    fr = d['final_results']
    if fr is None:
        continue
    gen = fr[fr['unit_id'] == GEN_ID].sort_values('hour')
    fp = d['final_bids']
    fp_gen = fp[fp['unit_id'] == GEN_ID].sort_values('hour') if fp is not None else None

    ax = axes[i]
    ax.bar(gen['hour'], gen['dispatch'], color=colors[i], alpha=0.6, label='Dispatched MW')
    if fp_gen is not None:
        ax.bar(gen['hour'], fp_gen['bid_volume'].values - gen['dispatch'].values,
               bottom=gen['dispatch'].values, color=colors[i], alpha=0.2,
               label='Offered but not dispatched')
    ax.axhline(300, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('Hour')
    ax.set_title(label)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Volume (MW)')
fig.suptitle(f'{GEN_ID} — Dispatch vs Offered Volume', fontsize=14)
plt.tight_layout()
plt.show()

## 6. LMP Impact — How Does Each Strategy Affect Market Prices?

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for i, (label, d) in enumerate(data.items()):
    fr = d['final_results']
    if fr is None:
        continue
    gen = fr[fr['unit_id'] == GEN_ID].sort_values('hour')
    ax.plot(gen['hour'], gen['accepted_price'], 'o-',
            color=colors[i], label=f'{label} LMP', ms=5)

ax.axhline(MC, color='gray', ls='--', alpha=0.5, label=f'gen_0 MC = ${MC:.0f}')
ax.axhline(50, color='red', ls=':', alpha=0.4, label='gen_1 MC = $50')
ax.axhline(20, color='green', ls=':', alpha=0.4, label='slack MC = $20')
ax.set_xlabel('Hour')
ax.set_ylabel('LMP at Bus 2 ($/MWh)')
ax.set_title('Market Price Impact — Final Policies')
ax.legend(fontsize=9)
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

## 7. Summary Table

In [ ]:
rows = []
for label, d in data.items():
    fp = d['final_bids']
    fr = d['final_results']
    if fp is None or fr is None:
        continue

    fp_gen = fp[fp['unit_id'] == GEN_ID]
    fr_gen = fr[fr['unit_id'] == GEN_ID]

    fr_gen = fr_gen.copy()
    fr_gen['profit'] = (fr_gen['accepted_price'] - fr_gen['marginal_cost']) * fr_gen['dispatch']

    # Peak hours (forecast >= MC — roughly hours 7-17)
    peak_mask = fp_gen['hour'].between(7, 17)
    offpeak_mask = ~peak_mask

    peak_fr = fr_gen[fr_gen['hour'].between(7, 17)]
    offpeak_fr = fr_gen[~fr_gen['hour'].between(7, 17)]

    rows.append({
        'Strategy': label.split('(')[0].strip(),
        'Avg Bid Price (peak)': f'${fp_gen[peak_mask]["bid_price"].mean():.1f}',
        'Avg Bid Price (off-peak)': f'${fp_gen[offpeak_mask]["bid_price"].mean():.1f}',
        'Avg Volume (peak)': f'{fp_gen[peak_mask]["bid_volume"].mean():.0f} MW',
        'Avg Volume (off-peak)': f'{fp_gen[offpeak_mask]["bid_volume"].mean():.0f} MW',
        'Peak Profit': f'${peak_fr["profit"].sum():,.0f}',
        'Off-Peak Profit': f'${offpeak_fr["profit"].sum():,.0f}',
        'Total Profit': f'${fr_gen["profit"].sum():,.0f}',
        'Avg LMP (peak)': f'${peak_fr["accepted_price"].mean():.1f}',
        'Dispatch Rate (peak)': f'{(peak_fr["dispatch"] > 0).mean():.0%}',
    })

summary_df = pd.DataFrame(rows).set_index('Strategy')
display(summary_df.T)

## 8. Training Reward Trajectories

Compare how quickly each strategy converges and the final reward level.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for i, (label, d) in enumerate(data.items()):
    results = d['results'][d['results']['unit_id'] == GEN_ID]
    # Total profit per episode as a proxy for reward
    ep_profit = results.copy()
    ep_profit['profit'] = (ep_profit['accepted_price'] - ep_profit['marginal_cost']) * ep_profit['dispatch']
    ep_totals = ep_profit.groupby('episode')['profit'].sum()

    # Smoothed with rolling window
    smoothed = ep_totals.rolling(window=10, min_periods=1).mean()
    ax.plot(smoothed.index, smoothed.values, color=colors[i], label=label, lw=2)
    ax.fill_between(ep_totals.index,
                    ep_totals.rolling(10, min_periods=1).quantile(0.25),
                    ep_totals.rolling(10, min_periods=1).quantile(0.75),
                    alpha=0.15, color=colors[i])

ax.set_xlabel('Episode')
ax.set_ylabel('Total Episode Profit ($)')
ax.set_title(f'{GEN_ID} — Training Profit Trajectory (10-episode rolling mean)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 9. Strategy Space Exploration

Scatter plot of all training bids in (price, volume) space, colored by episode.
Shows how differently the two strategies explore the action space.

In [ ]:
PLOT_HOUR = 12  # Peak hour for comparison

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for ax, (label, d) in zip(axes, data.items()):
    bids = d['bids'][(d['bids']['unit_id'] == GEN_ID) & (d['bids']['hour'] == PLOT_HOUR)]
    sc = ax.scatter(bids['bid_price'], bids['bid_volume'],
                    c=bids['episode'], cmap='viridis', s=15, alpha=0.6)

    # Final policy marker
    fp = d['final_bids']
    if fp is not None:
        fp_h = fp[(fp['unit_id'] == GEN_ID) & (fp['hour'] == PLOT_HOUR)]
        if len(fp_h) > 0:
            ax.scatter(fp_h['bid_price'], fp_h['bid_volume'],
                      marker='X', s=200, c='red', zorder=10, label='Final policy')

    ax.axvline(MC, color='gray', ls='--', alpha=0.5)
    ax.axvline(50, color='red', ls=':', alpha=0.4)
    ax.set_xlabel('Bid Price ($/MWh)')
    ax.set_title(f'{label}\nHour {PLOT_HOUR}')
    ax.legend(fontsize=8)
    plt.colorbar(sc, ax=ax, label='Episode')

axes[0].set_ylabel('Bid Volume (MW)')
fig.suptitle(f'{GEN_ID} — Exploration in (Price, Volume) Space', fontsize=14)
plt.tight_layout()
plt.show()